# Lab 3: Automatic Prompt Optimization with Amazon Bedrock

One important consideration when comparing and migrating between Foundation Models is that **your prompt templates likely perform better or worse for certain models**: Obscuring the models' true potential for your use-case.

Of course, manually optimizing different prompts for every model you're interested to evaluate would be a lot of work - which would limit the number of models you could explore.

Thankfully, [Advanced Prompt Optimization on Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/advanced-prompt-optimization-how.html) can automate this process for you!

## 🛠️ Environment Setup

If you haven't already, you'll need to install the required libraries for the workshop. See [../pyproject.toml](../pyproject.toml) for details.

We've commented-out the below cell because you should've already run the installation for [lab 1a](../lab1/Lab1a_-_Model_Selection_Framework.ipynb) - but if you need it, you can remove the `# ` and run the cell:

In [ ]:
# %pip install -e ..

> ⚠️ **Reminder**
> - You'll need to restart your notebook kernel (⤾ button in the toolbar above) *if you've already run* any of the code cells below before changing any library installations.
> - On SageMaker you might see an error the installation warning about conflicts due to pip's dependency resolution. As long as the installations themselves were successful, this can be ignored.

In [ ]:
%load_ext autoreload
%autoreload 2

from copy import deepcopy
from datetime import datetime
from io import StringIO
import json
import re

import boto3
import pandas as pd
from IPython.display import display, Markdown
from upath import UPath

import utils

bedrock = boto3.client("bedrock")

### S3 Bucket and IAM Role

Amazon Bedrock's Prompt Optimizer will read input datasets from and write outputs to [Amazon S3](https://aws.amazon.com/s3/), so you'll need a bucket set up to get started.

To allow the service to read your S3 input data, invoke foundation models, and write results, you'll also need to set up an *execution role* for the optimization jobs to assume.

Finally, the IAM identity **you're** using will need sufficient permissions to start and manage these jobs, and to pass the role you want them to use.

**In our workshop environment**, on Amazon SageMaker:
1. We'll use the [SageMaker default bucket](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning-ex-bucket.html) for storing data
2. We've already configured your SageMaker notebook environment to use an IAM role that:
    - Can be assumed by Amazon Bedrock (for the optimization jobs) as well as Amazon SageMaker (for the notebook)
    - Has permissions to access the default bucket
    - Has broad permissions to call Amazon Bedrock APIs

You can go ahead and run the code cell below to automatically retrieve this bucket and role.

> **In your own AWS account:** Refer to the [Advanced Prompt Optimization prerequisites](https://docs.aws.amazon.com/bedrock/latest/userguide/advanced-prompt-optimization-prereqs.html) in the Amazon Bedrock Developer Guide for more information on the permissions required to use this feature. You can set `bucket` name and `role` ARN manually, instead of using the auto-discovery code below.

In [ ]:
# Code to auto-discover default S3 Bucket and current execution Role
# (Skip this and set `role = "..."` and `bucket_name = "..."` manually, if not using SageMaker)

import sagemaker

bucket_name = None

role = sagemaker.get_execution_role()
print(f"Execution Role for jobs: {role}")

sm_sess = sagemaker.Session()
if bucket_name is None and sm_sess is not None:
    # set to default bucket if a bucket name is not given
    bucket_name = sm_sess.default_bucket()

sm_sess = sagemaker.Session(default_bucket=bucket_name)
print(f"Model Evaluation bucket: {bucket_name}")

Within your chosen S3 Bucket, we'll organize all this lab's data under a shared folder prefix to help keep things tidy.

This prefix should **NOT** have a trailing `/`:

In [ ]:
s3_prefix_base = "apo-demo"

# Subfolders for dataset inputs and result outputs:
s3_datasets_prefix = f"{s3_prefix_base}/datasets"
s3_results_prefix = f"{s3_prefix_base}/results"

For multi-modal datasets referencing objects like images or documents, these must be already uploaded to Amazon S3 and passed in by reference... so go we'll go ahead and upload the sample data that you can find in the [datasets/](datasets/) folder:

In [ ]:
!aws s3 sync datasets s3://{bucket_name}/{s3_datasets_prefix}

## How it works

**You provide:**

1. One to five target model(s) on Bedrock that you'd like to optimize for
2. Your current prompt template, with handlebars-style `{{placeholders}}` for the variables that are filled in at run-time
3. A dataset of samples including: 
    1. The input values to put in your template, and
    2. A "reference response" for the evaluator to refer to
4. Configuration for **how to evaluate** a response, which can be either:
    1. Default *(not recommended)* - A generic LLM judge evaluator... Or
    2. Up to five short "steering criteria" - from a single word like "professional" to a short sentence describing how responses should be... Or
    3. Full custom LLM-as-a-Judge configuration - including which LLM to use, and an evaluation prompt template that can include the `{{prompt}}`, `{{response}}`, and `{{referenceResponse}}` placeholders... Or
    4. A custom AWS Lambda function, which can implement whatever logic (or AI calls) you like - so long as it returns a score.

**Amazon Bedrock** works through your dataset and:

- Injects the variable values into your prompt template
- Invokes your target model(s) to generate response(s)
- Evaluates the response(s) by the criteria you defined
- Analyzes the results and automatically re-writes your template
- Repeats this feedback loop with advanced, proprietary strategies

**You get:**

1. 📄 Optimized prompt templates for your target model(s)
2. 📊 Evaluation scores for each evaluation sample
3. 🏎️ Latency metrics (time to first token, or TTFT) for each target model
4. 💲 Cost estimates for each target model

## Example: Steering Criteria with MathVista

[MathVista](https://mathvista.github.io/) is a published evaluation and dataset in which a vision-capable AI model is shown a chart, table, or diagram and asked a math/reasoning question.

In this example, we'll to steer the model to an **answer-first** response format with **LaTeX-rendered derivations** that explicitly reference the visual elements.

This is a good use-case for LLM-judged, steering-based evaluation: There's no easy numeric metric for "good explanation format", but the criteria are easy to state.

A small sample of the canonical dataset has already been prepared for you in Amazon Bedrock APO Job format, in the `datasets/` folder. Run the cell below to replace the template's `s3Asset` relative file links with actual `s3Uri`s to the Amazon S3 bucket where we uploaded the assets earlier. We'll print out the initial pre-optimization prompt template, and an example record from the dataset, for you to take a look at the format:

In [ ]:
dataset_id = "mathvista"

dataset = utils.resolve_sample_data_template(
    dataset_id=dataset_id,
    bucket_name=bucket_name,
    s3_datasets_prefix=s3_datasets_prefix,
)

print("INITIAL PROMPT TEMPLATE")
print("-----------------------")
print(dataset["promptTemplate"])
print("\n")
print("EXAMPLE DATASET ITEM")
print("--------------------")
print(json.dumps(dataset["evaluationSamples"][0], indent=2))

Next we'll add the configuration for how we'd like responses to be evaluated, to generate (and upload) the final input file for the optimization job.

In [ ]:
dataset["steeringCriteria"] = [
    "State the final answer first on its own line, in the format `Answer: <X>` (a number, an option letter, or short text).",
    "After the answer line, provide step-by-step mathematical derivations grounded in the image, with each derivation step rendered in LaTeX (`$...$` for inline math, `$$...$$` for display equations).",
    "Reasoning must reference specific visual elements of the image (axes, labels, shapes, colors, table cells, etc.) where relevant.",
    "Do not output any prose outside of the answer line and the LaTeX-formatted derivation steps.",
]

dataset_s3_uri = f"s3://{bucket_name}/{s3_datasets_prefix}/{dataset_id}/sample_steering.resolved.jsonl"

with UPath(dataset_s3_uri).open("w") as f:
    # Note this is actually a JSON-Lines file - so must be rendered all on one line:
    f.write(json.dumps(dataset))

print(f"Uploaded to {dataset_s3_uri}\n")
utils.print_dataset_without_samples(dataset)

With the full input file now prepared, including both the base dataset and the evaluation/steering criteria, we're ready to start our prompt optimization job.

For this example, we'll try preparing optimized templates for 3 different models:
- [Anthropic Claude Sonnet 4.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-4-5.html) - A proprietary model launched September 2025
- [Qwen3 VL 235B A22B](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-qwen-qwen3-vl-235b-a22b.html) - An open weight model launched September 2025
- [Moonshot AI Kimi K2.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-moonshot-ai-kimi-k2-5.html) - An open weight model launched January 2026

In [ ]:
job_name = f"demo-{dataset_id}-steering-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
output_s3uri = f"s3://{bucket_name}/{s3_results_prefix}/{job_name}/"  # (Must have trailing slash)

print(f"Starting job name {job_name} ...")
create_resp = bedrock.create_advanced_prompt_optimization_job(
    jobName=job_name,
    modelConfigurations=[
        {"modelId": "moonshotai.kimi-k2.5"},
        {"modelId": "qwen.qwen3-vl-235b-a22b"},
        {"modelId": "us.anthropic.claude-sonnet-4-6"},
    ],
    inputConfig={"s3Uri": dataset_s3_uri},
    outputConfig={"s3Uri": output_s3uri},
)
job_arn = create_resp["jobArn"]
print(f"Job ARN: {job_arn}")

<div style="background-color: #d4edda; border-left: 4px solid #28a745; padding: 15px; border-radius: 5px;">

<strong>⏰ This job will typically take around 70 minutes to complete...</strong><br><br>

...But for AWS-led events, we've **pre-run** some jobs already - so you shouldn't have to wait around.

Run the cell below to find the most recent already-completed job matching the name pattern.

ℹ️ If you're running this notebook in your own AWS Account without pre-run jobs, you can check on the status of your job in the [Amazon Bedrock Console](https://console.aws.amazon.com/bedrock/home?#/prompt-optimization) and directly set `completed_job_arn = job_arn` to the job you want to analyze.
</div>

In [ ]:
completed_job_arn = utils.find_apo_job(job_name=re.compile(".*mathvista.*"), job_status="Completed")
completed_job_id = completed_job_arn.rpartition("/")[2]
completed_job = bedrock.get_advanced_prompt_optimization_job(jobIdentifier=completed_job_arn)

display(Markdown("\n".join([
    "Got completed job ARN:",
    f"```\n{completed_job_arn}\n```",
    f"[Open in the Amazon Bedrock Console](https://console.aws.amazon.com/bedrock/home?#/prompt-optimization/{completed_job_id})",
    f"```json\n{json.dumps(completed_job, default=str, indent=2)}\n```",
])))

▶️ **Try following the link above** to view the job result in the Amazon Bedrock Console, which provides summary tables and graphs as well as per-example details from the dataset!

The job results will be saved to a subfolder created under your configured output S3 URI, named after the job's auto-generated unique ID.

Run the cell below to read the result JSON directly S3. We've also provided a utility function (in [utils.py](utils.py)) that creates a simpler flattened view of the complex result JSON.

In [ ]:
result_file_path = (
    UPath(completed_job["outputConfig"]["s3Uri"])
    / completed_job_id
    / "advanced_prompt_optimization_results.jsonl"
)
print(f"Fetching results from:\n{result_file_path.as_posix()}")
with result_file_path.open() as f:
    results = [json.loads(line) for line in f]
    results_flat = utils.flatten_results(results)
print(f"\n✅ Results fetched")

The flattened job summary includes average metrics (before and after optimization) on:
1. The evaluation scores,
2. The Time-To-First-Token (TTFT) latencies,
3. The number of input tokens consumed, and
4. The number of output tokens consumed

In the AWS Console, you also have the opportunity to enter pricing information to directly model how these different token counts correspond to different costs.

Run the cell below to visualize the job summary. You can edit the `drop` and `sort_values` settings to control which columns show and how the records (models) are sorted:

In [ ]:
pd.DataFrame(results_flat).drop(
    columns=["originalTemplate", "optimizedTemplate", "metricLabel", "failureReason"]
).sort_values(by="optimizedScore", ascending=False)

In our original test run, we saw evaluation metrics improve by between 10-20% during the optimization, with highest scoring model being Kimi K2.5 at ~90.3%. The optimization increased the average input length for all 3 models, but reduced the average output token count.

▶️ Your results may vary - did you observe similar?

There's lots more information in the detailed result data than shown in this summary. Run the cell below to compare the prompt templates before and after optimization, for each model. You can try exploring the `results` object to discover more.

In [ ]:
for row in results_flat:
    display(utils.render_prompt_diff(row, heading_md=f"### `{row['templateId']}` for {row['modelId']}"))

## Conclusion & Key Takeaways

In this notebook, we saw how optimizing your prompt templates can significantly improve output quality, depending on your model and target use-case... And how Amazon Bedrock's [Advanced Prompt Optimization feature](https://docs.aws.amazon.com/bedrock/latest/userguide/advanced-prompt-optimization-how.html) can do this optimization for you automatically - accelerating your ability to compare new models and find the best configuration(s) for your particular use-case.

These optimizations can often mean the difference between needing a higher-cost, frontier foundation model for a particular use-case - versus being able to achieve the same results with cheaper or faster open models.

One important call-out, is to make sure you're using the newer [Advanced Prompt Optimization](https://docs.aws.amazon.com/bedrock/latest/userguide/advanced-prompt-optimization-how.html) APIs. Amazon Bedrock also has an older [Simple Prompt Optimization API](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-management-optimize.html) which may still be applicable in some cases, but is less flexible and usually delivers less strong results.

For more examples covering other types of evaluation (including custom Lambda functions), see the [advanced-prompt-optimization folder](https://github.com/aws-samples/amazon-bedrock-samples/tree/main/advanced-prompt-optimization) in the Amazon Bedrock Samples GitHub repository.